# Basic 01 - Load and Inspect
Small, linear workflow: load -> summary -> contacts/publications -> pick study/assay/run -> load data with metadata.


## 1) Imports


In [1]:
from pathlib import Path
import sys
import logging
import warnings

import pandas as pd
from IPython.display import display


## 2) Quiet logs (optional)


In [2]:
warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)


## 3) Make local package importable (if not installed with `pip install -e .`)


In [3]:
def _ensure_local_package() -> None:
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    for root in search_roots:
        if (root / "isa_phm").is_dir() and (root / "pyproject.toml").exists():
            root_s = str(root)
            if root_s not in sys.path:
                sys.path.insert(0, root_s)
            return
    raise RuntimeError("Could not locate python-wrapper root with isa_phm package.")

_ensure_local_package()


## 4) Import wrapper


In [4]:
from isa_phm import ISAWrapper


## 5) Pick ISA JSON


In [5]:
# Example A: milling golden fixture
# ISA_JSON = Path(r"g:/ISA/ISA-PHM-Wizard/src/tests/fixtures/golden/isa-phm-out-milling.json")

# Example B: XJTU (uncomment to use)
ISA_JSON = Path(r"G:\ISA\Datasets\Wentelteef-Bearing-Diagnostic\Wentelteef Bearing Diagnlostic ISA-PHM.json")

ISA_JSON


WindowsPath('G:/ISA/Datasets/Wentelteef-Bearing-Diagnostic/Wentelteef Bearing Diagnlostic ISA-PHM.json')

## 6) Build wrapper


In [6]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=ISA_JSON.parent,
    strict_validation=False,
)


## 7) Summary


In [7]:
display(wrapper.summary())


,title,identifier,experiment_type,n_studies,n_contacts,n_publications,description_short,source_path
0,Bearing Diagnostic Test,16ecca9c-7284-4393-b5c8-89cbbebe0dea,prognostics-experiment,3,1,0,Test Dataset for Bearing Diagnostics on the We...,G:\ISA\Datasets\Wentelteef-Bearing-Diagnostic\...


## 8) Contacts


In [8]:
display(wrapper.contacts())


,contact_id,full_name,first_name,last_name,email,affiliation,roles,orcid
0,,Nathan Houwaart,Nathan,Houwaart,n.m.houwaart@hva.nl,Hogeschool van Amsterdam (HvA),Data curation,


## 9) Publications


In [9]:
display(wrapper.publications())


,title,doi,pubmed_id,status,author_tokens,corresponding_author,resolved_author_names,resolved_author_emails,unresolved_author_tokens


## 10) Pick first study


In [10]:
study_name = wrapper.list_studies()[0].title
study = wrapper.study(study_name)
study_name


'SKF6204 - RTF 1'

## 11) Pick first assay


In [11]:
assay_id = study.list_assays()[0].assay_id
assay = study.assay(assay_id)
assay_id


'a_st01_se01'

## 12) Pick first run


In [12]:
run_id = assay.list_runs()[0].run_id
run_id


'run_01'

## 13) Load one run with resolved file metadata


In [13]:
df, meta = assay.load_dataframe_with_meta(run_id=run_id, file_type="auto")


## 14) Show load metadata


In [14]:
display(pd.DataFrame([meta.model_dump()]))


,assay_id,run_id,requested_file_type,resolved_file_type,file_path,from_cache,csv_engine,csv_sep,csv_encoding,csv_detection_source,csv_bad_lines
0,a_st01_se01,run_01,auto,raw,G:\ISA\Datasets\Wentelteef-Bearing-Diagnostic\...,False,c,",",utf-8,common-separators,error


## 15) Show first rows


In [15]:
display(df.head(10))


,time,value
0,0.000000,1539.998481
1,0.000053,1539.991711
2,0.000105,1540.001333
3,0.000158,1539.999108
4,0.000211,1540.013649
5,0.000263,1540.005649
6,0.000316,1540.000317
7,0.000368,1539.996645
8,0.000421,1540.009188
9,0.000474,1539.996169


## 16) Optional: AI export


In [16]:
ctx = wrapper.ai_context(include_semantics=True, include_validation=False)
list(ctx.keys())


['schema_version',
 'generated_at_utc',
 'source_path',
 'investigation',
 'contacts',
 'publications',
 'studies',
 'assays',
 'factors',
 'semantic_manifest']

## 17) List AI tools


In [17]:
tools_df = pd.DataFrame(wrapper.list_tools())
display(tools_df)


,name,description,input_schema
0,ai_context,Return metadata-only normalized AI context.,"{'include_semantics': 'bool (default true)', '..."
1,load_dataframe_with_meta,Load one run and return load metadata plus JSO...,"{'study_id': 'str (required)', 'assay_id': 'st..."
2,validate_dataset,Run wrapper validation checks and return issue...,"{'check_files': 'bool (default true)', 'semant..."


## 18) Call tool: metadata context


In [18]:
tool_ctx = wrapper.call_tool("ai_context", {
    "include_semantics": False,
    "include_validation": False,
})
tool_ctx["ok"], sorted(tool_ctx["result"].keys())


(True,
 ['assays',
  'contacts',
  'factors',
  'generated_at_utc',
  'investigation',
  'publications',
  'schema_version',
  'source_path',
  'studies'])

## 19) Call tool: run data + load metadata (JSON-safe)


In [35]:
tool_df = wrapper.call_tool("load_dataframe_with_meta", {
    "study_id": study_name,
    "assay_id": assay_id,
    "run_id": run_id,
    "file_type": "auto",
    "head_rows": 5,
})
display(pd.DataFrame([tool_df["result"]["metadata"]]))
display(pd.DataFrame(tool_df["result"]["dataframe"]["head"]))


,assay_id,run_id,requested_file_type,resolved_file_type,file_path,from_cache,csv_engine,csv_sep,csv_encoding,csv_detection_source,csv_bad_lines
0,a_st01_se01,run_001,auto,raw,G:\XJTU-SY_Bearing_Datasets\Data_split_per_fil...,True,c,",",utf-8,common-separators,error


,time,value
0,0.000000,-0.396395
1,0.000039,-0.123107
2,0.000078,0.988841
3,0.000117,0.006676
4,0.000156,-1.074386
